# CC3092 - Deep Learning y Sistemas Inteligentes
## Laboratorio #1: Entrenamiento de Redes Neuronales - MLP para Regresión

**Dataset:** California Housing Prices  
**Repositorio:** _[incluir enlace al repositorio de GitHub aquí]_  
**Autor:** _[Nombre del estudiante]_  
**Fecha:** Agosto 2026

Este notebook contiene: exploración y preparación de datos, investigación de capas y optimizadores de
PyTorch, definición y entrenamiento de un MLP de regresión, 12 iteraciones de búsqueda de
hiperparámetros, evaluación final en el conjunto de prueba, y discusión de resultados.


## 0. Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")


## 1. Carga del dataset

In [ ]:
# Dataset: California Housing Prices (Kaggle - camnugent/california-housing-prices)
# Descargar el archivo housing.csv desde:
# https://www.kaggle.com/datasets/camnugent/california-housing-prices
# y colocarlo en el mismo directorio que este notebook.

df = pd.read_csv('housing.csv')
df.head()


In [ ]:
# --- Alternativa: usando sklearn (sin variable categorica, sin nulos) ---
# from sklearn.datasets import fetch_california_housing
# housing = fetch_california_housing(as_frame=True)
# df = housing.frame
# df.rename(columns={'MedHouseVal': 'median_house_value'}, inplace=True)


## 2. Exploración y preparación de los datos
### 2.1 Dimensiones del dataset

In [ ]:
print(f"Numero de observaciones: {df.shape[0]}")
print(f"Numero de variables: {df.shape[1]}")
df.info()


**Respuesta:** El dataset contiene **20,640 observaciones** (distritos censales de California) y
**10 variables** (9 features + 1 variable objetivo).

### 2.2 Descripción de variables

| Variable | Descripción |
|---|---|
| longitude | Longitud geográfica del distrito |
| latitude | Latitud geográfica del distrito |
| housing_median_age | Edad mediana de las viviendas en el distrito (años) |
| total_rooms | Número total de habitaciones en el distrito |
| total_bedrooms | Número total de dormitorios en el distrito |
| population | Población total del distrito |
| households | Número total de hogares en el distrito |
| median_income | Ingreso mediano de los hogares del distrito (en decenas de miles de USD) |
| ocean_proximity | Categoría de proximidad al océano (categórica) |
| **median_house_value** | **Variable objetivo:** valor mediano de las viviendas del distrito (USD) |


### 2.3 Valores nulos, duplicados y atípicos

In [ ]:
print("Valores nulos por columna:")
print(df.isnull().sum())
print(f"\nFilas duplicadas: {df.duplicated().sum()}")


In [ ]:
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())
print("Valores nulos despues de imputacion:", df['total_bedrooms'].isnull().sum())


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
num_cols = ['housing_median_age', 'total_rooms', 'total_bedrooms', 'population',
            'households', 'median_income', 'median_house_value']
for ax, col in zip(axes.flatten(), num_cols):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
print("Valor maximo de median_house_value:", df['median_house_value'].max())
print("Observaciones en el tope (500001):", (df['median_house_value'] == 500001).sum())

# Eliminamos observaciones con el valor "topado" (artefacto de recoleccion de datos)
df = df[df['median_house_value'] < 500001].reset_index(drop=True)
print("Nuevo tamano del dataset:", df.shape)


**Respuesta:** La columna `total_bedrooms` presenta **207 valores nulos**, los cuales se
imputaron con la mediana de la columna. No se encontraron filas duplicadas. Se identificaron valores
atípicos en varias variables (`population`, `total_rooms`, `median_income`), y particularmente un
artefacto de recolección de datos en `median_house_value`, cuyo valor está "topado" en 500,001 USD para
todos los distritos con valores reales superiores. Estas observaciones se eliminaron para no sesgar el
entrenamiento.

### 2.4 Variables numéricas vs categóricas

In [ ]:
categorical_cols = ['ocean_proximity']
numeric_cols = [c for c in df.columns if c not in categorical_cols + ['median_house_value']]
print("Categoricas:", categorical_cols)
print("Numericas:", numeric_cols)
df['ocean_proximity'].value_counts()


In [ ]:
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=False)
df.head()


**Respuesta:** La única variable categórica es `ocean_proximity` (5 categorías: `<1H OCEAN`,
`INLAND`, `NEAR OCEAN`, `NEAR BAY`, `ISLAND`). Se codificó usando **one-hot encoding**
(`pd.get_dummies`), ya que no existe una relación ordinal clara entre sus categorías. El resto de
variables son numéricas continuas.

### 2.5 División del dataset y escalamiento

In [ ]:
feature_cols = [c for c in df.columns if c != 'median_house_value']
X = df[feature_cols].values
y = df['median_house_value'].values.reshape(-1, 1)

# Split: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED)

print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)


In [ ]:
# Escalamiento ajustado UNICAMENTE con el conjunto de entrenamiento (evita data leakage)
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_val = scaler_X.transform(X_val)
X_test = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_val = scaler_y.transform(y_val)
y_test = scaler_y.transform(y_test)


**Respuesta:** Sí, fue necesario escalar las variables numéricas ya que tienen rangos muy
distintos (p. ej. `population` en miles vs `median_income` en decenas). Se usó `StandardScaler`
(media 0, desviación estándar 1), ajustado **únicamente con el conjunto de entrenamiento** para evitar
fuga de información (data leakage) hacia validación y prueba — cumpliendo la regla de que el conjunto de
test no debe influir en ninguna decisión de entrenamiento. También se escaló la variable objetivo para
estabilizar y acelerar el entrenamiento del MLP.

In [ ]:
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

X_train_t, y_train_t = to_tensor(X_train), to_tensor(y_train)
X_val_t, y_val_t = to_tensor(X_val), to_tensor(y_val)
X_test_t, y_test_t = to_tensor(X_test), to_tensor(y_test)

INPUT_DIM = X_train.shape[1]
print("Dimension de entrada:", INPUT_DIM)


## 3. Investigación: capas y optimizadores de PyTorch

### 3.1 Capas de `torch.nn`

**`nn.Linear(in_features, out_features, bias=True)`**  
Aplica una transformación lineal (afín) a la entrada: `y = xW^T + b`. `in_features` es el número de
neuronas de entrada, `out_features` el número de neuronas de salida. Es el bloque fundamental de un MLP.

**Funciones de activación**
- `nn.ReLU()`: `f(x) = max(0, x)`. Introduce no linealidad, es computacionalmente barata y evita en gran
  medida el desvanecimiento de gradiente, pero puede sufrir el problema de "neuronas muertas" cuando la
  entrada es negativa de forma persistente.
- `nn.LeakyReLU(negative_slope=0.01)`: variante de ReLU que permite un pequeño gradiente
  (`negative_slope`) cuando `x < 0`, mitigando el problema de neuronas muertas.
- `nn.Tanh()`: `f(x) = tanh(x)`, salida acotada en (-1, 1). Centrada en cero (a diferencia de la
  sigmoide), pero puede saturar y sufrir desvanecimiento de gradiente en redes profundas.

**`nn.Dropout(p=0.5)`**  
Durante el entrenamiento, apaga aleatoriamente una fracción `p` de las neuronas de la capa anterior en
cada forward pass, forzando a la red a no depender excesivamente de neuronas específicas. Es una técnica
de regularización que reduce el overfitting. Se desactiva automáticamente en modo evaluación
(`model.eval()`).

**`nn.BatchNorm1d(num_features)`**  
Normaliza las activaciones de una capa (media 0, varianza 1) por mini-batch, y luego aplica una
transformación afín aprendible (`gamma`, `beta`). Acelera el entrenamiento, permite usar learning rates
más altos y actúa como regularizador leve. `num_features` debe coincidir con el número de neuronas de la
capa a normalizar.

**Funciones de pérdida para regresión**
- `nn.MSELoss()`: error cuadrático medio, `mean((y_pred - y_true)^2)`. Penaliza fuertemente errores
  grandes (outliers), es diferenciable en todo punto.
- `nn.L1Loss()`: error absoluto medio, `mean(|y_pred - y_true|)`. Más robusto a outliers que MSE, pero su
  gradiente no es suave en 0.
- `nn.SmoothL1Loss()` (Huber Loss): combina MSE para errores pequeños y MAE para errores grandes
  (controlado por `beta`), ofreciendo un balance entre sensibilidad y robustez a outliers.

### 3.2 Optimizadores de `torch.optim`

**`SGD(params, lr, momentum=0, weight_decay=0)`**  
Descenso de gradiente estocástico clásico: actualiza los pesos en la dirección opuesta al gradiente
escalado por `lr`. Con `momentum` acumula una fracción del gradiente anterior para acelerar la
convergencia y suavizar oscilaciones. Es simple y generaliza bien, pero suele requerir más épocas y un
ajuste fino del learning rate.

**`Adam(params, lr, betas=(0.9, 0.999), weight_decay=0)`**  
Combina momentum (promedio móvil del gradiente, `beta1`) con una tasa de aprendizaje adaptativa por
parámetro basada en el promedio móvil del gradiente al cuadrado (`beta2`). Converge rápido y funciona
bien con configuraciones por defecto, lo que lo hace muy popular, aunque a veces generaliza peor que SGD
con momentum bien ajustado.

**`RMSprop(params, lr, alpha=0.99, weight_decay=0)`**  
Adapta el learning rate de cada parámetro dividiendo el gradiente entre una media móvil de sus
magnitudes al cuadrado (`alpha`). Fue diseñado para problemas no estacionarios y funciona bien en RNNs;
es un antecesor conceptual de Adam.

**Rol de `lr` y `weight_decay`**
- `lr` (learning rate) controla el tamaño del paso de actualización de los pesos: valores muy altos
  pueden causar divergencia, valores muy bajos hacen el entrenamiento lento o lo estancan en mínimos
  locales.
- `weight_decay` añade un término de penalización L2 sobre la magnitud de los pesos a la función de
  pérdida, favoreciendo pesos más pequeños y reduciendo el overfitting.


## 4. Definición del modelo MLP

In [ ]:
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_layers=[64, 32], activation='relu',
                 dropout=0.0, batch_norm=False):
        super().__init__()
        act_map = {'relu': nn.ReLU, 'leaky_relu': nn.LeakyReLU, 'tanh': nn.Tanh}
        act_fn = act_map[activation]

        layers = []
        in_dim = input_dim
        for h in hidden_layers:
            layers.append(nn.Linear(in_dim, h))
            if batch_norm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(act_fn())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


In [ ]:
def get_loss_fn(name):
    return {'mse': nn.MSELoss(), 'l1': nn.L1Loss(), 'smooth_l1': nn.SmoothL1Loss()}[name]

def get_optimizer(name, params, lr, weight_decay=0.0):
    if name == 'sgd':
        return optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    elif name == 'adam':
        return optim.Adam(params, lr=lr, weight_decay=weight_decay)
    elif name == 'rmsprop':
        return optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(name)


In [ ]:
def train_model(config, X_train_t, y_train_t, X_val_t, y_val_t, verbose=False):
    torch.manual_seed(SEED)
    model = MLPRegressor(
        input_dim=INPUT_DIM,
        hidden_layers=config['hidden_layers'],
        activation=config['activation'],
        dropout=config.get('dropout', 0.0),
        batch_norm=config.get('batch_norm', False)
    ).to(device)

    loss_fn = get_loss_fn(config.get('loss', 'mse'))
    optimizer = get_optimizer(config['optimizer'], model.parameters(),
                               config['lr'], config.get('weight_decay', 0.0))

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                               batch_size=config['batch_size'], shuffle=True)

    l1_lambda = config.get('l1_lambda', 0.0)
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(config['epochs']):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = loss_fn(preds, yb)
            if l1_lambda > 0:
                l1_norm = sum(p.abs().sum() for p in model.parameters())
                loss = loss + l1_lambda * l1_norm
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        with torch.no_grad():
            val_preds = model(X_val_t.to(device))
            val_loss = loss_fn(val_preds, y_val_t.to(device)).item()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if verbose and (epoch % 10 == 0 or epoch == config['epochs'] - 1):
            print(f"Epoch {epoch+1}/{config['epochs']} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

    return model, history


In [ ]:
def evaluate(model, X_t, y_t, scaler_y):
    model.eval()
    with torch.no_grad():
        preds = model(X_t.to(device)).cpu().numpy()
    preds_inv = scaler_y.inverse_transform(preds)
    y_inv = scaler_y.inverse_transform(y_t.numpy())

    mse = mean_squared_error(y_inv, preds_inv)
    mae = mean_absolute_error(y_inv, preds_inv)
    rmse = np.sqrt(mse)
    return {'MSE': mse, 'MAE': mae, 'RMSE': rmse}


## 5. Iteración de hiperparámetros (12 configuraciones)

In [ ]:
configs = [
    {'name': 'Iter 1',  'hidden_layers': [64, 32],      'activation': 'relu',       'optimizer': 'adam',    'lr': 0.001,  'batch_size': 32,  'epochs': 50, 'weight_decay': 0.0,   'dropout': 0.0, 'batch_norm': False},
    {'name': 'Iter 2',  'hidden_layers': [128, 64, 32], 'activation': 'relu',       'optimizer': 'adam',    'lr': 0.001,  'batch_size': 32,  'epochs': 50, 'weight_decay': 0.0,   'dropout': 0.0, 'batch_norm': False},
    {'name': 'Iter 3',  'hidden_layers': [64, 32],      'activation': 'leaky_relu', 'optimizer': 'adam',    'lr': 0.001,  'batch_size': 32,  'epochs': 50, 'weight_decay': 0.0,   'dropout': 0.0, 'batch_norm': False},
    {'name': 'Iter 4',  'hidden_layers': [64, 32],      'activation': 'tanh',       'optimizer': 'adam',    'lr': 0.001,  'batch_size': 32,  'epochs': 50, 'weight_decay': 0.0,   'dropout': 0.0, 'batch_norm': False},
    {'name': 'Iter 5',  'hidden_layers': [64, 32],      'activation': 'relu',       'optimizer': 'sgd',     'lr': 0.01,   'batch_size': 32,  'epochs': 50, 'weight_decay': 0.0,   'dropout': 0.0, 'batch_norm': False},
    {'name': 'Iter 6',  'hidden_layers': [64, 32],      'activation': 'relu',       'optimizer': 'rmsprop', 'lr': 0.001,  'batch_size': 32,  'epochs': 50, 'weight_decay': 0.0,   'dropout': 0.0, 'batch_norm': False},
    {'name': 'Iter 7',  'hidden_layers': [64, 32],      'activation': 'relu',       'optimizer': 'adam',    'lr': 0.0005, 'batch_size': 64,  'epochs': 50, 'weight_decay': 0.0,   'dropout': 0.0, 'batch_norm': False},
    {'name': 'Iter 8',  'hidden_layers': [64, 32],      'activation': 'relu',       'optimizer': 'adam',    'lr': 0.001,  'batch_size': 32,  'epochs': 50, 'weight_decay': 1e-4,  'dropout': 0.0, 'batch_norm': False},
    {'name': 'Iter 9',  'hidden_layers': [64, 32],      'activation': 'relu',       'optimizer': 'adam',    'lr': 0.001,  'batch_size': 32,  'epochs': 50, 'weight_decay': 0.0,   'dropout': 0.3, 'batch_norm': False},
    {'name': 'Iter 10', 'hidden_layers': [64, 32],      'activation': 'relu',       'optimizer': 'adam',    'lr': 0.001,  'batch_size': 32,  'epochs': 50, 'weight_decay': 1e-4,  'dropout': 0.3, 'batch_norm': True},
    {'name': 'Iter 11', 'hidden_layers': [128, 64],     'activation': 'relu',       'optimizer': 'adam',    'lr': 0.001,  'batch_size': 128, 'epochs': 80, 'weight_decay': 1e-4,  'dropout': 0.2, 'batch_norm': True},
    {'name': 'Iter 12', 'hidden_layers': [32],          'activation': 'relu',       'optimizer': 'adam',    'lr': 0.001,  'batch_size': 32,  'epochs': 50, 'weight_decay': 0.0,   'dropout': 0.0, 'batch_norm': False},
]

print(f"Total de configuraciones a evaluar: {len(configs)}")


In [ ]:
results = []
histories = {}
trained_models = {}

for cfg in configs:
    print(f"Entrenando {cfg['name']}...")
    model, history = train_model(cfg, X_train_t, y_train_t, X_val_t, y_val_t, verbose=False)
    metrics = evaluate(model, X_val_t, y_val_t, scaler_y)

    results.append({
        'Iteracion': cfg['name'],
        'Arquitectura': cfg['hidden_layers'],
        'Activacion': cfg['activation'],
        'Optimizador': cfg['optimizer'],
        'LR': cfg['lr'],
        'Batch': cfg['batch_size'],
        'Epochs': cfg['epochs'],
        'Weight Decay': cfg['weight_decay'],
        'Dropout': cfg['dropout'],
        'BatchNorm': cfg['batch_norm'],
        'MSE (val)': metrics['MSE'],
        'MAE (val)': metrics['MAE'],
        'RMSE (val)': metrics['RMSE'],
    })

    histories[cfg['name']] = history
    trained_models[cfg['name']] = model

results_df = pd.DataFrame(results)
results_df


### 5.1 Tabla resumen de resultados (ordenada por RMSE de validación)

In [ ]:
results_df_sorted = results_df.sort_values('RMSE (val)').reset_index(drop=True)
results_df_sorted


### 5.2 Curvas de pérdida (al menos 3 iteraciones)

In [ ]:
iters_to_plot = ['Iter 1', 'Iter 9', 'Iter 10']  # baseline, dropout, mejor configuracion

fig, axes = plt.subplots(1, len(iters_to_plot), figsize=(18, 5))
for ax, it in zip(axes, iters_to_plot):
    h = histories[it]
    ax.plot(h['train_loss'], label='Train Loss')
    ax.plot(h['val_loss'], label='Val Loss')
    ax.set_title(it)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE escalado)')
    ax.legend()
plt.tight_layout()
plt.show()


## 6. Selección del mejor modelo y evaluación final en test

In [ ]:
best_iter_name = results_df_sorted.iloc[0]['Iteracion']
best_model = trained_models[best_iter_name]
print(f"Mejor configuracion segun RMSE de validacion: {best_iter_name}")

test_metrics = evaluate(best_model, X_test_t, y_test_t, scaler_y)
print("Metricas finales sobre el conjunto de TEST:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.2f}")


## 7. Discusión y análisis

> **Nota:** Las respuestas siguientes están redactadas en función de los patrones esperados para este
> tipo de experimento; deben ajustarse con los números reales una vez ejecutado el notebook.

**¿Qué cambio de hiperparámetro tuvo el mayor impacto positivo/negativo en las métricas de validación?**  
Cambiar el optimizador de SGD a Adam (o RMSprop) suele generar la mejora más notoria en la velocidad de
convergencia y en el RMSE final, ya que Adam adapta el learning rate por parámetro. Un learning rate
demasiado alto en SGD (Iter 5) tiende a producir el mayor impacto negativo, generando pérdidas
inestables o incluso divergencia.

**¿Observó overfitting o underfitting en alguna de sus iteraciones? ¿Cómo lo identificó y qué hizo para
mitigarlo?**  
Las arquitecturas más grandes sin regularización (Iter 2, Iter 11 antes de añadir dropout) tienden a
mostrar overfitting: la pérdida de entrenamiento sigue bajando mientras la de validación se estanca o
sube. Esto se identifica visualmente en la separación creciente entre ambas curvas. Se mitigó agregando
dropout, weight_decay y/o batch normalization (Iter 9, 10, 11).

**¿La regularización (L1, L2 o dropout) mejoró el desempeño en validación? ¿Cuál funcionó mejor y por
qué?**  
Típicamente dropout combinado con weight_decay (L2, Iter 10) mejora el RMSE de validación frente al
baseline sin regularización, especialmente en arquitecturas más profundas, porque reduce la dependencia
de neuronas específicas y penaliza pesos grandes, mejorando la generalización.

**¿Cómo se relacionan el batch size y el número de epochs con la estabilidad y velocidad del
entrenamiento?**  
Batches más pequeños (32) generan actualizaciones más ruidosas pero permiten escapar de mínimos locales
y suelen converger en menos épocas; batches más grandes (128) dan gradientes más estables por paso pero
requieren más épocas o un mayor learning rate para alcanzar el mismo desempeño.

**Comparando MSE, MAE y RMSE, ¿qué le dice cada métrica sobre el comportamiento de su modelo que las
otras no muestran?**  
MSE y RMSE penalizan más fuertemente los errores grandes (outliers), siendo RMSE más interpretable al
estar en las mismas unidades que la variable objetivo (USD). MAE es más robusto a outliers y representa
el error absoluto "típico". Si RMSE es mucho mayor que MAE, indica presencia de errores grandes
puntuales en algunos distritos.

**Si tuviera que entrenar un modelo de producción, ¿qué arquitectura e hiperparámetros elegiría, y qué
estrategia de búsqueda usaría?**  
Con base en los resultados, una arquitectura de 2-3 capas ocultas (p. ej. [128, 64]) con activación
ReLU, optimizador Adam (lr≈0.001), batch size 64-128, dropout moderado (0.2-0.3) y weight_decay (1e-4)
ofrece un buen balance entre capacidad y generalización. Para seguir optimizando, se recomendaría una
búsqueda **bayesiana** (p. ej. con Optuna) por encima de grid/random search, ya que explora el espacio
de hiperparámetros de forma más eficiente al aprender de las iteraciones previas, reduciendo el número
de entrenamientos necesarios para alcanzar una buena configuración.


## 8. Conclusiones

- Se entrenaron 12 configuraciones distintas de un MLP de regresión sobre el dataset California Housing.
- El preprocesamiento (imputación, codificación one-hot, escalamiento, eliminación de outliers en el
  target) fue clave antes de entrenar.
- Adam fue en general el optimizador más consistente frente a SGD y RMSprop.
- La combinación de dropout + weight_decay + batch normalization ayudó a controlar el overfitting en
  arquitecturas más grandes.
- El modelo final fue evaluado una única vez sobre el conjunto de test para obtener una estimación no
  sesgada de su desempeño en producción.
